# Classifier-gated chat-format steering
Use a T4 GPU runtime. Build the minimal input archive locally with `python reproduce/build_gated_payload.py`. Inputs include frozen detector features; this does not benchmark live feature extraction. Only the required files are uploaded. Download results before ending the runtime.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.15.1', 'xgboost==3.4.1'])


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
name = next(iter(uploaded))
PAYLOAD = Path('/content/gated_chat_payload')
with zipfile.ZipFile(name) as archive:
    if any(Path(n).name != n for n in archive.namelist()): raise ValueError('Non-flat payload')
    archive.extractall(PAYLOAD)


In [ ]:
import importlib.util, json, shutil
from IPython.display import clear_output
spec=importlib.util.spec_from_file_location('gated_chat', PAYLOAD/'gated_chat.py')
runner=importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
OUT=Path('/content/gated_chat_run')
def progress(value):
    clear_output(wait=True)
    print(json.dumps(value), flush=True)
try:
    result=runner.main(PAYLOAD, OUT, callback=progress)
    print(json.dumps(result, indent=2))
finally:
    if OUT.exists(): shutil.make_archive('/content/gated_chat_results', 'zip', OUT)


In [ ]:
files.download('/content/gated_chat_results.zip')
